# 15.5 - Deepfakes & Misuse

**Module 15 - Ethics & Responsible AI** | Netsetos GenAI Engineering

This notebook works through Deepfakes & Misuse hands-on: Map your threat surface from your capabilities; Why detection loses twice; Provenance: prove what is real, not what is fake; Watermarking generated text with a green-list z-test; Rank the misuses with a risk matrix; Defense-in-depth: leaky layers that multiply.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

Before any of the risk math runs, the notebook sets a reproducibility note. Everything in this lesson is deterministic, keyless arithmetic - there is no model call, no API key, no random seed to fix at import time.

In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

A single comment-only cell. It flags that the worked examples use fixed, seeded values so every run reproduces the same numbers - there is nothing to install or import here.

**How the code works - step by step**
- A lone comment noting the lesson uses seeded / fixed values throughout, so outputs are reproducible.
- No packages, no imports - `import math` appears later, inside the watermarking cell, right where it is needed.

**In one line:** a reproducibility note, not executable setup.

**What you'll see:** (no output - environment prep)

## 1 - Map your threat surface from your capabilities

You cannot defend a harm you have not listed. Before shipping a generative feature, enumerate the abuse categories and tag each with the modality it needs, then see which ones YOUR product actually unlocks. The insight: your threat surface is defined by what you can generate, the same way an attack surface is defined by what a system exposes.

In [ ]:
# THE THREAT SURFACE: generative models make convincing fakes across modalities, and YOUR capabilities decide which harms you
# enable. Map each harm to the modality it needs, then see which ones your product unlocks. Adding a modality widens the surface.
HARMS = [  # (harm, modalities it needs (any-of), the real-world harm)
    ("non-consensual intimate imagery", {"image"},          "the most-reported deepfake abuse"),
    ("voice-clone fraud (vishing)",     {"audio"},          "fastest-growing: fake-CEO and family-emergency scams"),
    ("political disinformation",        {"image", "video"}, "erodes trust; election interference"),
    ("impersonation / KYC bypass",      {"image", "audio"}, "defeats face and voice identity checks"),
    ("fabricated evidence",             {"image", "video"}, "fake documents, fake footage"),
    ("deepfake-video blackmail",        {"video"},          "coercion with fabricated video")]
product_caps = {"text", "image", "audio"}       # our product generates these; NOT video
print("Product capabilities: {}".format(sorted(product_caps)))
enabled = 0
for harm, needs, why in HARMS:
    ok = bool(needs & product_caps)             # enabled if we can produce ANY modality the harm needs
    print("  [{}] {:<34} needs {:<16} {}".format("X" if ok else " ", harm, "/".join(sorted(needs)), why))
    if ok: enabled += 1
print()
print("{} of {} harm categories are ENABLED by this product's capabilities.".format(enabled, len(HARMS)))
print("Your threat surface is defined by what you can generate - adding video would unlock the last one. Map it before you ship.")

# Output:
# Product capabilities: ['audio', 'image', 'text']
#   [X] non-consensual intimate imagery    needs image            the most-reported deepfake abuse
#   [X] voice-clone fraud (vishing)        needs audio            fastest-growing: fake-CEO and family-emergency scams
#   [X] political disinformation           needs image/video      erodes trust; election interference
#   [X] impersonation / KYC bypass         needs audio/image      defeats face and voice identity checks
#   [X] fabricated evidence                needs image/video      fake documents, fake footage
#   [ ] deepfake-video blackmail           needs video            coercion with fabricated video
#
# 5 of 6 harm categories are ENABLED by this product's capabilities.
# Your threat surface is defined by what you can generate - adding video would unlock the last one. Map it before you ship.

**Explanation**

This is an inventory harness, not a model call. It holds a table of six known synthetic-media harms, each tagged with the modalities that would enable it, and intersects that against a fixed set of product capabilities to count which harms are in reach. The point is to make the pre-launch mapping mechanical: capabilities in, enabled-harm list out.

**How the code works - step by step**
- **`HARMS`** - a list of six `(harm, needs, why)` tuples; `needs` is a set of modalities, any one of which enables the harm.
- **`product_caps`** - the fixed capability set `{text, image, audio}` (deliberately NOT video).
- **The loop** - for each harm, `needs & product_caps` is a set intersection; a non-empty intersection means the harm is enabled, marked `[X]`, and increments `enabled`.
- **The tally** - prints how many of the six harms this capability set unlocks.

**In one line:** intersect each harm's required modalities with your capabilities -> count what you enable.

**What you'll see:** It prints the capability list, then each harm marked `[X]` (enabled) or `[ ]` (not), and the tally: 5 of 6 harm categories are enabled - only deepfake-video blackmail (video-only) stays out of reach until you add video.

## 2 - Why detection loses twice

The reflex is to run a classifier that flags fakes. This cell shows the two independent reasons that cannot be your strategy: accuracy decays as generators improve, and at a realistic base rate even a good detector drowns in false alarms.

In [ ]:
# THE DETECTION ARMS RACE + THE BASE-RATE PROBLEM: deepfake detectors chase artifacts, and every better generator erases them,
# so accuracy decays. Worse, at a realistic base rate (few fakes among many real files) even a 'good' detector floods false alarms.
detector_acc = {"generator v1 (2023)": 0.95, "generator v2 (2026)": 0.62}   # same detector, newer generator
print("Same detector, accuracy vs generator vintage:")
for gen, acc in detector_acc.items():
    print("  {:<22} {:.0%}".format(gen, acc))
print()
# Base-rate: 10,000 media items, 1% are fake. Detector: true-positive rate 0.90, false-positive rate 0.10.
N, base_rate, tpr, fpr = 10000, 0.01, 0.90, 0.10
fakes = int(N * base_rate); real = N - fakes
tp = int(fakes * tpr); fp = int(real * fpr)
precision = tp / (tp + fp)
print("Of {:,} items, {} are fake. Detector flags {} true fakes and {} real items (TPR {:.0%}, FPR {:.0%}).".format(N, fakes, tp, fp, tpr, fpr))
print("precision = TP / (TP + FP) = {} / {} = {:.1%} -> {:.0%} of everything it flags is a FALSE alarm.".format(tp, tp + fp, precision, 1 - precision))
print("Detection decays as generators improve AND drowns in false positives at low base rates. You cannot detect your way out -")
print("the durable answer is PROVENANCE: prove what is real at the source, rather than trying to spot every fake after the fact.")

# Output:
# Same detector, accuracy vs generator vintage:
#   generator v1 (2023)    95%
#   generator v2 (2026)    62%
#
# Of 10,000 items, 100 are fake. Detector flags 90 true fakes and 990 real items (TPR 90%, FPR 10%).
# precision = TP / (TP + FP) = 90 / 1080 = 8.3% -> 92% of everything it flags is a FALSE alarm.
# Detection decays as generators improve AND drowns in false positives at low base rates. You cannot detect your way out -
# the durable answer is PROVENANCE: prove what is real at the source, rather than trying to spot every fake after the fact.

**Explanation**

A two-part measurement, not a detector. The first part is a lookup table showing the same detector's accuracy against an old versus a new generator; the second computes a confusion matrix from a base rate and derives precision. Together they quantify the arms race and the base-rate problem in a few lines of arithmetic.

**How the code works - step by step**
- **`detector_acc`** - the same detector scored against two generator vintages (0.95 on 2023, 0.62 on 2026), printed to show accuracy decays.
- **Base-rate setup** - `N=10000` items, `base_rate=0.01`, `tpr=0.90`, `fpr=0.10`; splits into `fakes` and `real`.
- **`tp` / `fp`** - true positives are fakes correctly flagged (`fakes * tpr`), false positives are real items wrongly flagged (`real * fpr`).
- **`precision = tp / (tp + fp)`** - the fraction of everything flagged that is actually fake.

**In one line:** accuracy decays vintage-to-vintage, and low base rate crushes precision, so most flags are false alarms.

**What you'll see:** It prints the accuracy drop (95% -> 62%), then that among 10,000 items with 100 fakes the detector flags 90 true fakes plus 990 real items, giving precision = 90/1080 = about 8.3% - meaning 92% of everything it flags is a false alarm.

## 3 - Provenance: prove what is real, not what is fake

Since you cannot spot fakes at scale, flip the question. C2PA / Content Credentials cryptographically signs media at each step - capture, edit, export - into a tamper-evident chain, so verification is a signature check rather than a pixel analysis. This cell builds that chain and then tampers with it.

In [ ]:
# PROVENANCE (C2PA / Content Credentials): instead of spotting fakes, cryptographically SIGN media at each step - capture, edit,
# export - so anyone can verify an unbroken chain back to a trusted source. A stripped or tampered step breaks the chain.
def sign(payload, prev_sig):                    # toy deterministic 'signature' (a real C2PA manifest uses public-key crypto)
    return sum(ord(c) for c in payload) * 31 + prev_sig
def verify_chain(chain):
    prev = 0
    for step in chain:
        expected = sign(step["payload"], prev)
        if step["sig"] != expected:
            return False, step["by"]
        prev = expected
    return True, None
GOOD = [{"by": "camera",   "payload": "capture:raw",  "sig": None},
        {"by": "editor",   "payload": "edit:crop",    "sig": None},
        {"by": "exporter", "payload": "export:jpeg",  "sig": None}]
prev = 0                                         # build a valid manifest chain
for s in GOOD:
    s["sig"] = sign(s["payload"], prev); prev = s["sig"]
ok, _ = verify_chain(GOOD)
print("Signed capture->edit->export chain: {}".format("VERIFIED (provenance intact)" if ok else "UNVERIFIED"))
TAMPERED = [dict(s) for s in GOOD]
TAMPERED[1] = {"by": "unknown tool", "payload": "edit:deepfake-face", "sig": TAMPERED[1]["sig"]}  # swapped content, old sig
ok2, broke = verify_chain(TAMPERED)
print("After an untrusted tool swaps the edit: {} (chain breaks at '{}')".format("VERIFIED" if ok2 else "UNVERIFIED", broke))
print()
print("Provenance flips the question from 'is this fake?' to 'can you prove it is real?'. C2PA / Content Credentials signs the")
print("chain; a missing or broken manifest is itself the signal. It is the answer detection cannot be - but only if signing is widespread.")

# Output:
# Signed capture->edit->export chain: VERIFIED (provenance intact)
# After an untrusted tool swaps the edit: UNVERIFIED (chain breaks at 'unknown tool')
#
# Provenance flips the question from 'is this fake?' to 'can you prove it is real?'. C2PA / Content Credentials signs the
# chain; a missing or broken manifest is itself the signal. It is the answer detection cannot be - but only if signing is widespread.

**Explanation**

A miniature provenance validator. It uses a toy deterministic `sign` function (real C2PA uses public-key crypto) so each step's signature depends on both its own payload and the previous signature, chaining them. `verify_chain` walks the chain and recomputes each signature; the demo shows a clean chain verifying and a swapped step breaking it.

**How the code works - step by step**
- **`sign(payload, prev_sig)`** - a stand-in signature: a character sum times 31 plus the previous signature, so each link binds to the one before it.
- **`verify_chain(chain)`** - recomputes the expected signature for every step; returns `False` and the offending actor at the first mismatch.
- **`GOOD`** - a camera -> editor -> exporter chain; the loop signs each step against the running `prev`, producing a valid manifest.
- **`TAMPERED`** - copies the chain, then replaces the edit step with a deepfake payload while keeping the old signature, so its recomputed signature no longer matches.

**In one line:** each step signs the previous one, so verifying is walking the chain - and any swap breaks it.

**What you'll see:** It prints the clean chain as VERIFIED (provenance intact), then after an untrusted tool swaps the edit it prints UNVERIFIED, naming the link where the chain breaks ('unknown tool').

## 4 - Watermarking generated text with a green-list z-test

Provenance covers media that carries a manifest; watermarking embeds a signal directly into content you generate. The standard text approach (Kirchenbauer / SynthID-Text) biases the model toward a pseudo-random 'green' subset of the vocabulary, then a z-test measures whether green tokens appear far more often than chance.

In [ ]:
# WATERMARKING generated text (statistical / green-list, as in SynthID-style schemes): at each step the model prefers a pseudo-
# random 'green' subset of the vocabulary. Human text hits green at the base rate; watermarked text hits it far more - a z-test detects it.
import math
GAMMA = 0.25                                     # green list is a quarter of the vocabulary
def z_score(green, n):                           # how many standard deviations above the chance rate
    expected = n * GAMMA
    sd = math.sqrt(n * GAMMA * (1 - GAMMA))
    return round((green - expected) / sd, 2)
N = 60; THRESH = 4.0                             # z above 4 is overwhelming evidence of a watermark
cases = [("watermarked output", 40), ("human-written text", 16), ("watermarked, then paraphrased", 22)]
for label, green in cases:
    z = z_score(green, N)
    verdict = "WATERMARK detected" if z >= THRESH else "no watermark"
    print("{:<32} {}/{} green tokens  z={:>5}  -> {}".format(label, green, N, z, verdict))
print()
print("The watermark is invisible to a reader but statistically loud (z=7.45 vs 0.30 for human text). The catch: paraphrasing")
print("rewrites enough tokens to drop z below the threshold, so a watermark is a strong signal on unmodified output, not tamper-proof.")

# Output:
# watermarked output               40/60 green tokens  z= 7.45  -> WATERMARK detected
# human-written text               16/60 green tokens  z=  0.3  -> no watermark
# watermarked, then paraphrased    22/60 green tokens  z= 2.09  -> no watermark
#
# The watermark is invisible to a reader but statistically loud (z=7.45 vs 0.30 for human text). The catch: paraphrasing
# rewrites enough tokens to drop z below the threshold, so a watermark is a strong signal on unmodified output, not tamper-proof.

**Explanation**

A statistical detector, not a generator. It computes a z-score - how many standard deviations the observed green-token count sits above the chance rate - for three cases: watermarked, human, and paraphrased. The takeaway is that the watermark is invisible to a reader but statistically loud, yet paraphrasing rewrites enough tokens to drop the score below threshold.

**How the code works - step by step**
- **`GAMMA = 0.25`** - the green list is a quarter of the vocabulary, so human text hits green about 25% of the time by chance.
- **`z_score(green, n)`** - expected green is `n*GAMMA`, standard deviation is `sqrt(n*GAMMA*(1-GAMMA))`; z is `(green - expected)/sd`, rounded.
- **`N=60`, `THRESH=4.0`** - a 60-token sample; a z at or above 4 is overwhelming evidence of a watermark.
- **`cases`** - three green counts (40 watermarked, 16 human, 22 paraphrased); the loop scores each and labels it detected or not.

**In one line:** count green tokens, convert to a z-score, and flag anything far above the chance rate.

**What you'll see:** It prints each case with its green count, z-score, and verdict: watermarked 40/60 -> z=7.45 (detected), human 16/60 -> z=0.30 (no watermark), and watermarked-then-paraphrased 22/60 -> z=2.09 (below threshold, watermark removed).

## 5 - Rank the misuses with a risk matrix

You have a threat surface (Step 1); now prioritize it, because you cannot mitigate everything equally before launch. Score each harm on likelihood and severity, multiply for a single risk number, and mitigate everything above a threshold - the same likelihood-times-severity method a safety engineer uses for any hazard.

In [ ]:
# THE MISUSE RISK MATRIX: before you ship, score each harm by LIKELIHOOD (how easy/probable) and SEVERITY (how bad), multiply
# for a risk score, and mitigate everything above a threshold. This turns 'could this be misused?' into a ranked, actionable list.
RISKS = [  # (harm, likelihood 1-5, severity 1-5)
    ("voice-clone fraud",       4, 5),
    ("political disinformation", 4, 4),
    ("non-consensual imagery",   3, 5),
    ("impersonation / KYC bypass", 3, 4),
    ("fabricated evidence",      2, 4),
    ("reputation / defamation",  3, 3)]
THRESHOLD = 12                                    # risk >= 12 must be mitigated before release
scored = sorted(((L * S, h, L, S) for h, L, S in RISKS), reverse=True)
print("Misuse risk matrix (risk = likelihood x severity, threshold {}):".format(THRESHOLD))
must_mitigate = 0
for risk, h, L, S in scored:
    tag = "MITIGATE" if risk >= THRESHOLD else "monitor"
    if risk >= THRESHOLD: must_mitigate += 1
    print("  {:<28} L{} x S{} = {:>2}   {}".format(h, L, S, risk, tag))
print()
print("{} of {} harms cross the mitigation threshold and cannot ship without a control.".format(must_mitigate, len(RISKS)))
print("Likelihood x severity ranks where to spend effort: the fake-CEO voice scam (20) outranks defamation (9). Score, then mitigate.")

# Output:
# Misuse risk matrix (risk = likelihood x severity, threshold 12):
#   voice-clone fraud            L4 x S5 = 20   MITIGATE
#   political disinformation     L4 x S4 = 16   MITIGATE
#   non-consensual imagery       L3 x S5 = 15   MITIGATE
#   impersonation / KYC bypass   L3 x S4 = 12   MITIGATE
#   reputation / defamation      L3 x S3 =  9   monitor
#   fabricated evidence          L2 x S4 =  8   monitor
#
# 4 of 6 harms cross the mitigation threshold and cannot ship without a control.
# Likelihood x severity ranks where to spend effort: the fake-CEO voice scam (20) outranks defamation (9). Score, then mitigate.

**Explanation**

A prioritization harness. It scores six harms on two 1-to-5 axes, multiplies them into a risk score, sorts descending, and counts how many cross a mitigation threshold. It does two jobs at once: forces you to enumerate the misuses, and ranks them so limited pre-launch effort goes where the risk is highest.

**How the code works - step by step**
- **`RISKS`** - six `(harm, likelihood, severity)` tuples, each axis on a 1-5 scale.
- **`THRESHOLD = 12`** - any risk score at or above 12 must be mitigated before release.
- **`scored`** - a generator that computes `L*S` per harm, sorted descending so the highest risk prints first.
- **The loop** - tags each harm MITIGATE or monitor against the threshold and counts `must_mitigate`.

**In one line:** likelihood times severity, sort, and mitigate everything above the line.

**What you'll see:** It prints the ranked matrix - voice-clone fraud (L4xS5=20) at the top, down through defamation (9) and fabricated evidence (8) - and reports that 4 of 6 harms cross the threshold of 12 and cannot ship without a control.

## 6 - Defense-in-depth: leaky layers that multiply

No single guardrail stops misuse, and 'the filter isn't perfect' is never a reason to skip a layer. Stack independent guardrails so an abusive request has to defeat all of them - the miss rates multiply, turning several mediocre filters into one strong net (the Swiss-cheese model).

In [ ]:
# DEFENSE-IN-DEPTH: no single guardrail stops misuse, so you stack independent layers. An abusive request slips through only if
# EVERY layer misses it, so the miss rates MULTIPLY - even mediocre layers combine into a strong net. (The Swiss-cheese model.)
LAYERS = [  # (layer, miss rate = fraction of abuse it lets through)
    ("input prompt filter",          0.40),
    ("output content classifier",    0.30),
    ("provenance + watermark",       0.50),
    ("rate-limit + KYC on high-risk", 0.60),
    ("human review of flagged",      0.25)]
combined_miss = 1.0
print("Layered defenses (an abusive request must beat ALL of them):")
for name, miss in LAYERS:
    print("  {:<32} lets through {:.0%}".format(name, miss))
    combined_miss *= miss
best_single = min(m for _, m in LAYERS)
print()
print("best SINGLE layer catches {:.0%} - not enough on its own.".format(1 - best_single))
print("STACKED, misses multiply: combined miss = {:.3f} -> the stack catches {:.1%} of abuse.".format(combined_miss, 1 - combined_miss))
print("No guardrail is perfect, so layer them: independent defenses turn several leaky filters into one that rarely leaks.")

# Output:
# Layered defenses (an abusive request must beat ALL of them):
#   input prompt filter              lets through 40%
#   output content classifier        lets through 30%
#   provenance + watermark           lets through 50%
#   rate-limit + KYC on high-risk    lets through 60%
#   human review of flagged          lets through 25%
#
# best SINGLE layer catches 75% - not enough on its own.
# STACKED, misses multiply: combined miss = 0.009 -> the stack catches 99.1% of abuse.
# No guardrail is perfect, so layer them: independent defenses turn several leaky filters into one that rarely leaks.

**Explanation**

A reliability calculation pointed at abuse. Each layer has an independent miss rate - the fraction of abuse it lets through. Because a request slips all the way through only if every layer misses it, the combined miss rate is the product of the individual ones, the same multiplication used for reliability in Module 12.

**How the code works - step by step**
- **`LAYERS`** - five `(layer, miss_rate)` guardrails: input filter, output classifier, provenance+watermark, rate-limit+KYC, human review.
- **The loop** - prints each layer's pass-through rate and multiplies `combined_miss` by every miss rate.
- **`best_single`** - the smallest miss rate, so `1 - best_single` is the best any single layer achieves.
- **The result** - `1 - combined_miss` is the stacked catch rate.

**In one line:** multiply the independent miss rates - the product is tiny even when each layer is leaky.

**What you'll see:** It prints each layer's pass-through rate (25% to 60%), then that the best single layer catches only 75%, but stacked the misses multiply to a combined miss of 0.009 - so the stack catches 99.1% of abuse.

## 7 - The disclosure and incident release gate

Transparency is now law: the EU AI Act Article 50 requires AI-generated content and deepfakes to be labelled, and a responsible provider needs a takedown-and-notify path before an incident, not after. This cell turns the whole toolkit into a release gate that fails safe.

In [ ]:
# DISCLOSURE + THE INCIDENT GATE (EU AI Act Art. 50): 2026 transparency law requires that AI-generated content and deepfakes be
# LABELLED, and that you can respond when abuse happens. Gate the release on disclosure, provenance, takedown, reporting, and sign-off.
CHECKS = {                                        # each must hold to ship a synthetic-media feature
    "outputs carry provenance / watermark": True,
    "AI-generated content is disclosed / labelled (EU AI Act Art. 50)": False,   # UNMET
    "takedown + victim-notification process exists": True,
    "misuse reporting channel is live": True,
    "red-team sign-off on misuse risks": True}
unmet = [name for name, ok in CHECKS.items() if not ok]
print("SYNTHETIC-MEDIA RELEASE GATE:")
for name, ok in CHECKS.items():
    print("  [{}] {}".format("x" if ok else " ", name))
print()
print("gate: {}".format("PASS - ship" if not unmet else "BLOCK - {} unmet:".format(len(unmet))))
for u in unmet:
    print("   - " + u)
print("Disclosure is now law, not courtesy: the EU AI Act (Art. 50) requires labelling AI-generated media and deepfakes, and a")
print("provider needs a takedown-and-notify path BEFORE an incident, not after. Transparency plus a response plan is the release bar.")

# Output:
# SYNTHETIC-MEDIA RELEASE GATE:
#   [x] outputs carry provenance / watermark
#   [ ] AI-generated content is disclosed / labelled (EU AI Act Art. 50)
#   [x] takedown + victim-notification process exists
#   [x] misuse reporting channel is live
#   [x] red-team sign-off on misuse risks
#
# gate: BLOCK - 1 unmet:
#    - AI-generated content is disclosed / labelled (EU AI Act Art. 50)
# Disclosure is now law, not courtesy: the EU AI Act (Art. 50) requires labelling AI-generated media and deepfakes, and a
# provider needs a takedown-and-notify path BEFORE an incident, not after. Transparency plus a response plan is the release bar.

**Explanation**

A release-gate evaluator. It holds a dict of five ship conditions - covering both the engineering duties (provenance, takedown, reporting, sign-off) and the legal one (disclosure) - and blocks the launch if any is unmet. The worked example deliberately leaves the disclosure label False to show the gate blocking on an Article 50 gap.

**How the code works - step by step**
- **`CHECKS`** - five boolean conditions that must all hold to ship; the disclosure/labelling check is set `False`.
- **`unmet`** - a list comprehension collecting every check whose value is False.
- **The report loop** - prints each check as `[x]` or `[ ]`.
- **The verdict** - PASS if `unmet` is empty, otherwise BLOCK, listing each unmet check by name.

**In one line:** all checks must pass to ship - one unmet condition blocks the release.

**What you'll see:** It prints the five-item gate with the disclosure label unchecked, then BLOCK - 1 unmet, naming the missing 'AI-generated content is disclosed / labelled (EU AI Act Art. 50)' check as the reason the feature cannot ship.

Read as one pipeline, the seven cells make a single argument: you cannot detect your way out of synthetic-media misuse, so you flip the question from "is this fake?" to "can I prove this is real?" You map which harms your capabilities enable, watch a detector lose to both the arms race and the base rate, verify a signed C2PA provenance chain, run a z-test on a green-list watermark (and watch paraphrasing wash it out), rank misuses by likelihood times severity, multiply leaky guardrails into a strong net, and gate the release on an EU AI Act Article 50 disclosure label. Every cell is keyless arithmetic - no model call, no API key. This is defensive engineering, not legal advice: the next lesson (15.6) turns these controls into the full regulatory regime, and 15.7 hands the misuse assessment and incident process to a governance program.